In [1]:
import jax.numpy as jnp
import numpy as np
import jax
import matplotlib.pyplot as plt

In [210]:
def _kalman_update(
    mean_prev: jnp.ndarray,
    cov_prev: jnp.ndarray,
    obs: jnp.ndarray,
    transition_matrix: jnp.ndarray,
    process_cov: jnp.ndarray,
    measurement_matrix: jnp.ndarray,
    measurement_cov: jnp.ndarray,
):
    """

    Parameters
    ----------
    mean_prev : jnp.ndarray, shape (n_states,)
        Previous state mean.
    cov_prev : jnp.ndarray, shape (n_states, n_states)
        Previous state covariance.
    obs : jnp.ndarray, shape (n_obs_dim,)
        Data observation.
    transition_matrix : jnp.ndarray, shape (n_states, n_states)
        State transition matrix.
    process_cov : jnp.ndarray, shape (n_states, n_states)
        State noise covariance.
    measurement_matrix : jnp.ndarray, shape (n_obs_dim, n_states)
        Maps the observation to the state
    measurement_cov : jnp.ndarray, shape (n_obs_dim, n_obs_dim)
        Observation noise covariance.
    """
    jax.debug.print("mean_prev: {}", mean_prev)
    # jax.debug.print("cov_prev: {}", cov_prev)
    # jax.debug.print("obs: {}", obs)
    # jax.debug.print("transition_matrix: {}", transition_matrix)
    # jax.debug.print("process_cov: {}", process_cov)
    # jax.debug.print("measurement_matrix: {}", measurement_matrix)
    # jax.debug.print("measurement_cov: {}", measurement_cov)

    # One step prediction
    one_step_mean = transition_matrix @ mean_prev
    one_step_cov = transition_matrix @ cov_prev @ transition_matrix.T + process_cov

    # Measurement update
    obs_mean = measurement_matrix @ one_step_mean
    # obs_cross_cov = one_step_cov @ measurement_matrix.T
    obs_cov = measurement_matrix @ one_step_cov @ measurement_matrix.T + measurement_cov

    residual_error = obs - obs_mean  # innovation
    kalman_gain = jax.scipy.linalg.solve(
        obs_cov, measurement_matrix @ one_step_cov, assume_a="pos"
    ).T

    posterior_mean = one_step_mean + kalman_gain @ residual_error
    posterior_cov = one_step_cov - kalman_gain @ obs_cov @ kalman_gain.T

    marginal_log_likelihood = jax.scipy.stats.multivariate_normal.logpdf(
        x=residual_error, mean=jnp.zeros_like(residual_error), cov=obs_cov
    )
    # cross_variance = (jnp.identity(mean_prev.shape[0]) - kalman_gain @ measurement_matrix) @ transition_matrix @ cov_prev

    return posterior_mean, posterior_cov, kalman_gain, marginal_log_likelihood


def _collapse_mixture_of_gaussians(
    means: jnp.ndarray,
    covariances: jnp.ndarray,
    weights: jnp.ndarray,
    n_gaussians: int = 1,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Approximate mixture of Gaussians with a mixture of n_gaussians Gaussians.

    Generalized Pseudo-Bayesian algorithm for moment matching in Gaussian mixtures.

    Parameters
    ----------
    means : jnp.ndarray, shape (n_states,)
        Mean of the Gaussian mixture.
    covariances : jnp.ndarray, shape (n_states, n_states)
        Covariance of the Gaussian mixture.
    weights : jnp.ndarray, shape (n_states,)
        Weights of the Gaussian mixture.
    n_gaussians : int, optional
        Number of Gaussians to approximate the mixture with, by default 1.

    Returns
    -------
    collapsed_means : jnp.ndarray
        Means of the approximated Gaussian mixture.
    collapsed_covariances : jnp.ndarray
        Covariances of the approximated Gaussian mixture.

    """
    # means are the average weighted means
    collapsed_means = weights @ means
    mean_diff = collapsed_means - means
    collapsed_covs = weights @ covariances + weights @ jnp.outer(mean_diff, mean_diff)

    return collapsed_means, collapsed_covs


def switching_kalman_filter(
    init_mean: jnp.ndarray,
    init_cov: jnp.ndarray,
    init_discrete_state_prob: jnp.ndarray,
    obs: jnp.ndarray,
    discrete_transition_matrix: jnp.ndarray,
    continuous_transition_matrix: jnp.ndarray,
    process_cov: jnp.ndarray,
    measurement_matrix: jnp.ndarray,
    measurement_cov: jnp.ndarray,
):
    """Switching Kalman filter for a linear Gaussian state space model with discrete states.

    Parameters
    ----------
    init_mean : jnp.ndarray, shape (n_cont_states,)
        Initial value of the continuous latent state
    init_cov : jnp.ndarray, shape (n_cont_states, n_cont_states)
        Initial covariance of the continuous latent state
    init_discrete_state_prob : jnp.ndarray, shape (n_discrete_states,)
        Initial probability of the discrete states
    obs : jnp.ndarray, shape (n_time, n_obs_dim)
        Observations
    discrete_transition_matrix : jnp.ndarray, shape (n_discrete_states, n_discrete_states)
        Transition matrix for the discrete states
    continuous_transition_matrix : jnp.ndarray, shape (n_cont_states, n_cont_states, n_discrete_states)
        Transition matrix for the continuous states
    process_cov : jnp.ndarray, shape (n_cont_states, n_cont_states, n_discrete_states)
        Process noise covariance
    measurement_matrix : jnp.ndarray, shape (n_obs_dims, n_cont_states, n_discrete_states)
        Map observations to the continuous states
    measurement_cov : jnp.ndarray, shape (n_obs_dims, n_obs_dims, n_discrete_states)
        Measurement variance

    Returns
    -------
    _type_
        _description_
    """

    n_discrete_states = discrete_transition_matrix.shape[0]
    n_cont_states = continuous_transition_matrix.shape[0]

    def _step(carry, obs_t):
        mean_prev, cov_prev, discrete_state_prob_prev = carry

        # mean_prev, shape (n_cont_states, n_discrete_states)
        # cov_prev, shape (n_cont_states, n_cont_states, n_discrete_states)

        # joint prob between time steps
        joint_discrete_state_prob = jnp.zeros((n_discrete_states, n_discrete_states))
        posterior_mean = jnp.zeros(
            (n_discrete_states, n_discrete_states, n_cont_states)
        )
        posterior_cov = jnp.zeros(
            (n_discrete_states, n_discrete_states, n_cont_states, n_cont_states)
        )
        kalman_gain = jnp.zeros(
            (n_discrete_states, n_discrete_states, n_cont_states, n_cont_states)
        )
        marginal_log_likelihood = jnp.zeros((n_discrete_states, n_discrete_states))

        # Kalman update for each pair of discrete states
        # vmap twice over the discrete states
        for state_j in range(n_discrete_states):
            for state_i in range(n_discrete_states):
                (
                    posterior_mean_ij,
                    posterior_cov_ij,
                    kalman_gain_ij,
                    marginal_log_likelihood_ij,
                ) = _kalman_update(
                    mean_prev[:, state_i],
                    cov_prev[:, :, state_i],
                    obs_t,
                    continuous_transition_matrix[:, :, state_j],
                    process_cov[:, :, state_j],
                    measurement_matrix[:, :, state_j],
                    measurement_cov[:, :, state_j],
                )
                posterior_mean.at[state_i, state_j].set(posterior_mean_ij)
                posterior_cov.at[state_i, state_j].set(posterior_cov_ij)
                kalman_gain.at[state_i, state_j].set(kalman_gain_ij)
                marginal_log_likelihood.at[state_i, state_j].set(
                    marginal_log_likelihood_ij
                )

                joint_discrete_state_prob.at[state_i, state_j].set(
                    jnp.exp(marginal_log_likelihood_ij)
                    * discrete_transition_matrix[state_i, state_j]
                    * discrete_state_prob_prev[state_i]
                )

        # Need to make sure the likelihood is normalized to max 1 for numerical stability

        # Compute the marginal probability of the discrete states
        joint_discrete_state_prob /= jnp.sum(joint_discrete_state_prob)
        discrete_state_prob = jnp.sum(joint_discrete_state_prob, axis=0, keepdims=True)
        discrete_state_weights = joint_discrete_state_prob / discrete_state_prob

        # Compute the collapsed Gaussian
        collapsed_mean, collapsed_cov, collapsed_weights = (
            _collapse_mixture_of_gaussians(
                posterior_mean_ij,
                posterior_cov_ij,
                discrete_state_weights,
                n_gaussians=1,
            )
        )

        return (collapsed_mean, collapsed_cov, discrete_state_prob), (
            collapsed_mean,
            collapsed_cov,
            discrete_state_prob,
        )

    (_, _, _), (filter_mean, filter_cov, filter_discrete_state_prob) = jax.lax.scan(
        _step,
        (init_mean, init_cov, init_discrete_state_prob),
        obs,
    )

    return filter_mean, filter_cov, filter_discrete_state_prob

In [99]:
n_obs_dims = 2
n_cont_states = 3
n_discrete_states = 4

mean_prev = np.random.rand(n_cont_states, n_discrete_states)
cov_prev = np.random.rand(n_cont_states, n_cont_states, n_discrete_states)
obs_t = np.random.rand(n_obs_dims)
continuous_transition_matrix = np.random.rand(
    n_cont_states, n_cont_states, n_discrete_states
)
process_cov = np.random.rand(n_cont_states, n_cont_states, n_discrete_states)
measurement_matrix = np.random.rand(n_obs_dims, n_cont_states, n_discrete_states)
measurement_cov = np.random.rand(n_obs_dims, n_obs_dims, n_discrete_states)


posterior_mean = jnp.zeros((n_discrete_states, n_discrete_states, n_cont_states))
posterior_cov = jnp.zeros(
    (n_discrete_states, n_discrete_states, n_cont_states, n_cont_states)
)
kalman_gain = jnp.zeros(
    (n_discrete_states, n_discrete_states, n_cont_states, n_obs_dims)
)
marginal_log_likelihood = jnp.zeros((n_discrete_states, n_discrete_states))

for state_j in range(n_discrete_states):
    for state_i in range(n_discrete_states):
        (
            posterior_mean_ij,
            posterior_cov_ij,
            kalman_gain_ij,
            marginal_log_likelihood_ij,
        ) = _kalman_update(
            mean_prev[:, state_i],
            cov_prev[:, :, state_i],
            obs_t,
            continuous_transition_matrix[:, :, state_j],
            process_cov[:, :, state_j],
            measurement_matrix[:, :, state_j],
            measurement_cov[:, :, state_j],
        )
        posterior_mean = posterior_mean.at[state_i, state_j, :].set(posterior_mean_ij)
        posterior_cov = posterior_cov.at[state_i, state_j].set(posterior_cov_ij)
        kalman_gain = kalman_gain.at[state_i, state_j].set(kalman_gain_ij)
        marginal_log_likelihood = marginal_log_likelihood.at[state_i, state_j].set(
            marginal_log_likelihood_ij
        )

In [ ]:
(
    posterior_mean_ij,
    posterior_cov_ij,
    kalman_gain_ij,
    marginal_log_likelihood_ij,
) = _kalman_update(
    mean_prev[:, state_i],
    cov_prev[:, :, state_i],
    obs_t,
    continuous_transition_matrix[:, :, state_j],
    process_cov[:, :, state_j],
    measurement_matrix[:, :, state_j],
    measurement_cov[:, :, state_j],
)

In [125]:
from functools import partial

vmap_posterior_mean = jnp.zeros((n_discrete_states, n_discrete_states, n_cont_states))
vmap_posterior_cov = jnp.zeros(
    (n_discrete_states, n_discrete_states, n_cont_states, n_cont_states)
)
vmap_kalman_gain = jnp.zeros(
    (n_discrete_states, n_discrete_states, n_cont_states, n_obs_dims)
)
vmap_marginal_log_likelihood = jnp.zeros((n_discrete_states, n_discrete_states))

for state_j in range(n_discrete_states):
    vmapped_kalman_update_partial = partial(
        _kalman_update,
        obs=obs_t,
        transition_matrix=continuous_transition_matrix[:, :, state_j],
        process_cov=process_cov[:, :, state_j],
        measurement_matrix=measurement_matrix[:, :, state_j],
        measurement_cov=measurement_cov[:, :, state_j],
    )
    vmapped_kalman_update = jax.vmap(vmapped_kalman_update_partial, in_axes=(1, 2))
    (
        posterior_mean_j,
        posterior_cov_j,
        kalman_gain_j,
        marginal_log_likelihood_j,
    )  = vmapped_kalman_update(mean_prev, cov_prev)

    vmap_posterior_mean = vmap_posterior_mean.at[:, state_j, :].set(posterior_mean_j)
    vmap_posterior_cov = vmap_posterior_cov.at[:, state_j].set(posterior_cov_j)
    vmap_kalman_gain = vmap_kalman_gain.at[:, state_j].set(kalman_gain_j)
    vmap_marginal_log_likelihood = vmap_marginal_log_likelihood.at[:, state_j].set(
        marginal_log_likelihood_j
    )

(
    np.allclose(posterior_mean, vmap_posterior_mean, equal_nan=True),
    np.allclose(posterior_cov, vmap_posterior_cov, equal_nan=True),
    np.allclose(kalman_gain, vmap_kalman_gain, equal_nan=True),
    np.allclose(marginal_log_likelihood, vmap_marginal_log_likelihood, equal_nan=True),
)

(False, False, False, True)

In [198]:
state_j = 0

for state_i in range(n_discrete_states):
    (
        posterior_mean_ij,
        posterior_cov_ij,
        kalman_gain_ij,
        marginal_log_likelihood_ij,
    ) = _kalman_update(
        mean_prev[:, state_i],
        cov_prev[:, :, state_i],
        obs_t,
        continuous_transition_matrix[:, :, state_j],
        process_cov[:, :, state_j],
        measurement_matrix[:, :, state_j],
        measurement_cov[:, :, state_j],
    )

    print(posterior_mean_ij)

[0.13028811 0.1987052  0.23928793]
[0.14305806 0.17299974 0.67555845]
[0.31255633 0.19359863 0.38195014]
[0.11057711 0.05398607 0.54119754]


In [209]:
vmapped_kalman_update(mean_prev[:, [0]], cov_prev[:, :, [0]])[0]

Array([[0.12427703, 0.20582242, 0.22663008]], dtype=float32)

In [205]:
mean_prev[:, [0]].shape

(3, 1)

In [203]:
vmapped_kalman_update_partial(mean_prev[:, 0], cov_prev[:, :, 0])[0]

Array([0.13028811, 0.1987052 , 0.23928793], dtype=float32)

In [196]:
state_j = 0

vmapped_kalman_update_partial = partial(
    _kalman_update,
    obs=obs_t,
    transition_matrix=continuous_transition_matrix[:, :, state_j],
    process_cov=process_cov[:, :, state_j],
    measurement_matrix=measurement_matrix[:, :, state_j],
    measurement_cov=measurement_cov[:, :, state_j],
)
vmapped_kalman_update = jax.vmap(vmapped_kalman_update_partial, in_axes=(1, 2))
(
    posterior_mean_j,
    posterior_cov_j,
    kalman_gain_j,
    marginal_log_likelihood_j,
)  = vmapped_kalman_update(mean_prev, cov_prev)

posterior_mean_j


Array([[0.12427703, 0.20582242, 0.22663008],
       [0.16706264, 0.23030913, 0.67446387],
       [0.3156731 , 0.20591271, 0.37598407],
       [0.106659  , 0.05658275, 0.5316262 ]], dtype=float32)

In [116]:
vmapped_kalman_update = partial(
    _kalman_update,
    transition_matrix=continuous_transition_matrix[:, :, state_j],
    process_cov=process_cov[:, :, state_j],
    measurement_matrix=measurement_matrix[:, :, state_j],
    measurement_cov=measurement_cov[:, :, state_j],
)

vmapped_kalman_update, mean_prev.shape

(functools.partial(<function _kalman_update at 0x1868db250>, transition_matrix=array([[0.67205705, 0.46496917, 0.95017173],
        [0.20324787, 0.51594597, 0.14091597],
        [0.21362187, 0.42700726, 0.27016121]]), process_cov=array([[0.43103613, 0.69027437, 0.47944072],
        [0.75603677, 0.9196685 , 0.18183179],
        [0.23226232, 0.38305831, 0.72425164]]), measurement_matrix=array([[0.57250088, 0.09668837, 0.1114529 ],
        [0.05411453, 0.62966765, 0.20241808]]), measurement_cov=array([[0.60469048, 0.85238565],
        [0.65876465, 0.17453182]])),
 (3, 4))

In [113]:
posterior_mean[0], vmap_posterior_mean[0]

(Array([[ 0.13028811,  0.1987052 ,  0.23928793],
        [-0.217715  , -0.05090405, -0.15870646],
        [ 0.18856311,  0.21518339,  0.05493009],
        [        nan,         nan,         nan]], dtype=float32),
 Array([[0.12427703, 0.20582242, 0.22663008],
        [0.07056665, 0.08661755, 0.0578814 ],
        [       nan,        nan,        nan],
        [       nan,        nan,        nan]], dtype=float32))

In [130]:
vmapped_kalman_update = jax.vmap(
    jax.vmap(_kalman_update, in_axes=(None, None, None, 2, 2, 2, 2)),
    in_axes=(1, 2, None, None, None, None, None),
)

(
    vmap_posterior_mean,
    vmap_posterior_cov,
    vmap_kalman_gain,
    vmap_marginal_log_likelihood,
) = vmapped_kalman_update(
    mean_prev,
    cov_prev,
    obs_t,
    continuous_transition_matrix,
    process_cov,
    measurement_matrix,
    measurement_cov,
)

(
    np.allclose(posterior_mean, vmap_posterior_mean, equal_nan=True),
    np.allclose(posterior_cov, vmap_posterior_cov, equal_nan=True),
    np.allclose(kalman_gain, vmap_kalman_gain, equal_nan=True),
    np.allclose(marginal_log_likelihood, vmap_marginal_log_likelihood, equal_nan=True),
)

(False, False, False, True)

In [11]:
n_states = 3

numerator = np.random.rand(n_states, n_states)

joint_discrete_state_prob = numerator / jnp.sum(numerator)
discrete_state_prob = jnp.sum(joint_discrete_state_prob, axis=0)

discrete_state_weights = joint_discrete_state_prob / discrete_state_prob[None, :]

discrete_state_weights, discrete_state_prob

(Array([[0.02005424, 0.37786967, 0.27610204],
        [0.6235135 , 0.50655204, 0.14056215],
        [0.35643226, 0.1155783 , 0.5833358 ]], dtype=float32),
 Array([0.24918205, 0.44963327, 0.30118465], dtype=float32))

In [12]:
t = 0
n_discrete_states = 3
T = 1

numr = numerator[:, :, None]
W_norm = np.zeros((T + 1, 1))
W_ij = np.zeros((n_discrete_states, n_discrete_states, T + 1))
W_j = np.zeros((T + 1, n_discrete_states))

W_norm[t] = np.sum(numr[:, :, t])

# compute W_ij
W_ij[:, :, t] = numr[:, :, t] / W_norm[t]

# W_j = P(St=j|y_1:t)
for j in range(n_discrete_states):
    W_j[t, j] = np.sum(W_ij[:, j, t])

# g_ij = P(St-1=i|St=j,y_1:t) = weights of state components
g = np.zeros((n_discrete_states, n_discrete_states))
for j in range(n_discrete_states):
    for i in range(n_discrete_states):
        g[i, j] = W_ij[i, j, t] / W_j[t, j]

In [15]:
np.allclose(g, discrete_state_weights), np.allclose(W_j[t], discrete_state_prob)

(True, True)

In [23]:
n_cont_states = 2

x_ij = np.zeros(
    (n_cont_states, n_discrete_states, n_discrete_states)
)  # posterior state est obtained by filtering
V_ij = np.zeros(
    (n_cont_states, n_cont_states, n_discrete_states, n_discrete_states, T + 1)
)  # posterior error covariance matrix est
X = np.zeros(
    (n_cont_states, n_discrete_states, T + 1)
)  # new state X_j obatained by collapsing x_ij

V = np.zeros(
    (n_cont_states, n_cont_states, n_discrete_states, T + 1)
)  # new cov V_j obatained by collapsing V_ij


x_ij[:, :, :] = np.random.rand(n_cont_states, n_discrete_states, n_discrete_states)
V_ij[:, :, :, :, t] = np.random.rand(
    n_cont_states, n_cont_states, n_discrete_states, n_discrete_states
)


# approximate (using COLLAPSE - moment matching) new state
for j in range(n_discrete_states):
    X[:, j, t] = x_ij[:, :, j] @ g[:, j]

    V[:, :, j, t] = np.zeros((n_cont_states, n_cont_states))

    for i in range(n_discrete_states):
        m = x_ij[:, i, [j]] - X[:, [j], t]
        V[:, :, j, t] = V[:, :, j, t] + g[i, j] * (V_ij[:, :, i, j, t] + m @ m.T)

In [24]:
V[..., t].shape

(2, 2, 3)

In [25]:
X[..., t].shape

(2, 3)

In [27]:
X.shape

(2, 3, 2)

In [30]:
x_ij.shape, g.shape

((2, 3, 3), (3, 3))

In [38]:
(x_ij[:, :, j] @ g[:, j])

array([0.17957664, 0.68572116])

In [39]:
jnp.einsum("ij,j->i", x_ij[:, :, j], g[:, j])

Array([0.17957664, 0.68572116], dtype=float32)

In [69]:
def collapse_MoG_cross(
    conditional_means_x: jnp.ndarray,
    conditional_means_y: jnp.ndarray,
    conditional_cross_cov: jnp.ndarray,
    mixing_weights: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Collapse a mixture of Gaussians.

    Parameters
    ----------
    conditional_means_x : jnp.ndarray, shape (n_dims, n_discrete_states)
        E[X | S = j]
    conditional_means_y : jnp.ndarray, shape (n_dims, n_discrete_states)
        E[Y | S = j]
    conditional_cross_cov : jnp.ndarray, shape (n_dims, n_dims, n_discrete_states)
        E[X,Y | S = j]
    mixing_weights : jnp.ndarray, shape (n_discrete_states,)
        P[S = j]

    Returns
    -------
    unconditional_mean_x : jnp.ndarray, shape (n_dims,)
        E[X]
    unconditional_mean_y : jnp.ndarray, shape (n_dims,)
        E[Y]
    unconditional_cov_xy : jnp.ndarray, shape (n_dims, n_dims)
        E[X,Y]
    """

    unconditional_mean_x = conditional_means_x @ mixing_weights  # E[X]
    unconditional_mean_y = conditional_means_y @ mixing_weights  # E[Y]

    diff_x = conditional_means_x - unconditional_mean_x[:, None]
    diff_y = conditional_means_y - unconditional_mean_y[:, None]

    unconditional_cov_xy = (
        conditional_cross_cov @ mixing_weights + (diff_x * mixing_weights) @ diff_y.T
    )  # E[XY]

    return unconditional_mean_x, unconditional_mean_y, unconditional_cov_xy


conditional_means_x = np.random.rand(n_cont_states, n_discrete_states)  # E[X | S = j]
conditional_means_y = np.random.rand(n_cont_states, n_discrete_states)  # E[Y | S = j]
conditional_cross_cov = np.random.rand(
    n_cont_states, n_cont_states, n_discrete_states
)  # E[XY | S = j]
mixing_weights = np.random.rand(n_discrete_states)  # P[S = j]

collapse_MoG_cross(
    conditional_means_x, conditional_means_y, conditional_cross_cov, mixing_weights
)

(array([0.09587608, 0.09646295]),
 array([0.18714417, 0.04607859]),
 array([[0.09611016, 0.28386601],
        [0.21054627, 0.19259278]]))

In [71]:
@jax.jit
def collapse_MoG(
    conditional_means_x: jnp.ndarray,
    conditional_cov: jnp.ndarray,
    mixing_weights: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Collapse a mixture of Gaussians.

    Parameters
    ----------
    conditional_means_x : jnp.ndarray, shape (n_dims, n_discrete_states)
        E[X | S = j]
    conditional_cov : jnp.ndarray, shape (n_dims, n_dims, n_discrete_states)
        Cov[X | S = j]
    mixing_weights : jnp.ndarray, shape (n_discrete_states,)
        P[S = j]

    Returns
    -------
    unconditional_mean_x : jnp.ndarray, shape (n_dims,)
        E[X]
    unconditional_cov_x : jnp.ndarray, shape (n_dims, n_dims)
        Cov[X]
    """
    unconditional_mean_x = conditional_means_x @ mixing_weights  # E[X]
    diff_x = conditional_means_x - unconditional_mean_x[:, None]

    unconditional_cov_xx = (
        conditional_cov @ mixing_weights + (diff_x * mixing_weights) @ diff_x.T
    )  # E[XX]

    return unconditional_mean_x, unconditional_cov_xx


collapse_MoG(conditional_means_x, conditional_cross_cov, mixing_weights)

(Array([0.09587608, 0.09646294], dtype=float32),
 Array([[0.07342777, 0.29915968],
        [0.18195537, 0.21310604]], dtype=float32))

In [72]:
jax.vmap(collapse_MoG, in_axes=(0, 0, 0))(
    conditional_means_x, conditional_cross_cov, mixing_weights
)

ValueError: vmap got inconsistent sizes for array axes to be mapped:
  * most axes (2 of them) had size 2, e.g. axis 0 of argument conditional_means_x of type float32[2,3];
  * one axis had size 3: axis 0 of argument mixing_weights of type float32[3]